## Implementing GPT model from scratch.


In [24]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,  # Vocabulary size
    "context_length": 1024,  # Context length
    "emb_dim": 768,  # Embedding dimension
    "n_heads": 12,  # Number of attention heads
    "n_layers": 12,  # Number of layers
    "drop_rate": 0.1,  # Dropout rate
    "qkv_bias": False,  # Query-Key-Value bias
}

In [25]:
from torch import nn
import torch


class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        # d_out lai num_heads le divide garna milnu parxa
        # each head ko equal dimension ko lagi

        self.d_out = d_out

        self.num_heads = num_heads

        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # query projection layer banako

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # key projection layer banako

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # value projection layer banako

        self.out_proj = nn.Linear(d_out, d_out)
        # sabai heads combine pachi final projection garna use hunxa

        self.dropout = nn.Dropout(dropout)
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # causal mask banako
        # future tokens block garna use hunxa

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, token count, input dimension nikaleko

        keys = self.W_key(x)
        # input lai key vectors ma transform gareko

        queries = self.W_query(x)
        # input lai query vectors ma transform gareko

        values = self.W_value(x)
        # input lai value vectors ma transform gareko

        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # keys lai multiple heads ma split gareko

        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        # values lai multiple heads ma split gareko

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # queries lai multiple heads ma split gareko

        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)

        keys = keys.transpose(1, 2)
        # token ra head dimension swap gareko

        queries = queries.transpose(1, 2)
        # queries ko dimensions swap gareko

        values = values.transpose(1, 2)
        # values ko dimensions swap gareko

        attn_scores = queries @ keys.transpose(2, 3)
        # each head ko query ra key bich attention score nikaleko

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # required token size anusar mask select gareko

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # future token positions lai -inf banako
        # softmax pachi probability 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax attention apply gareko

        attn_weights = self.dropout(attn_weights)
        # attention weights ma dropout apply gareko

        context_vec = (attn_weights @ values).transpose(1, 2)
        # weighted sum garera context vector banako
        # transpose garera original order ma lyako

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # sabai heads combine garera reshape gareko

        context_vec = self.out_proj(context_vec)
        # final linear projection apply gareko

        return context_vec
        # final multi-head attention output return gareko

In [26]:
from torch import nn


class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"],
        )
        # multi-head attention layer banako

        self.ff = FeedForward(cfg)
        # feed forward network banako

        self.norm1 = LayerNorm(cfg["emb_dim"])
        # attention block agadi layer normalization

        self.norm2 = LayerNorm(cfg["emb_dim"])
        # feed forward block agadi layer normalization

        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        # shortcut connection ma dropout apply garna

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        # original input lai shortcut ma save gareko

        x = self.norm1(x)
        # attention block agadi normalization apply gareko

        x = self.att(x)
        # multi-head attention apply gareko
        # output shape [batch_size, num_tokens, emb_size]

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        # Shortcut connection for feed forward block
        shortcut = x
        # attention output lai shortcut ma save gareko

        x = self.norm2(x)
        # feed forward block agadi normalization apply gareko

        x = self.ff(x)
        # feed forward network apply gareko

        x = self.drop_shortcut(x)
        # dropout apply gareko

        x = x + shortcut
        # residual connection: original input add gareko

        return x
        # final transformer block output return gareko

In [27]:
import torch
import torch.nn as nn


# GELU activation function banako class
class GELU(nn.Module):

    # class initialize garna use huncha
    def __init__(self):
        # parent nn.Module ko constructor call gareko
        super().__init__()

    # forward pass define gareko
    def forward(self, x):

        # GELU formula apply gareko
        return (
            0.5  # formula ko constant value
            * x  # input value multiply gareko
            * (
                1
                + torch.tanh(  # tanh activation use gareko
                    # sqrt(2/pi) calculate gareko
                    torch.sqrt(torch.tensor(2.0 / torch.pi))
                    # x + 0.044715*x^3 calculate gareko
                    * (x + 0.044715 * torch.pow(x, 3))
                )
            )
        )

In [28]:
from torch import nn


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [29]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [30]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


# ya bata mai gpt model ko code ho mathi ko needed function jun previously banako theo


In [31]:
from torch import nn
import torch


class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        # token embedding layer banako (word lai vector ma convert garna)

        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        # positional embedding layer banako (sequence position encode garna)

        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        # embeddings ma dropout apply garna (regularization ko lagi)

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        # multiple transformer blocks stack gareko (n_layers anusar)

        self.final_norm = LayerNorm(cfg["emb_dim"])
        # final layer normalization apply garna

        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        # output linear layer (embedding → vocab logits)

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        # input shape: batch size ra sequence length nikaleko

        tok_embeds = self.tok_emb(in_idx)
        # input tokens lai embedding vectors ma convert gareko

        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        # sequence positions ko embedding banako

        x = tok_embeds + pos_embeds
        # token embedding + positional embedding combine gareko
        # shape: [batch_size, num_tokens, emb_size]

        x = self.drop_emb(x)
        # embeddings ma dropout apply gareko

        x = self.trf_blocks(x)
        # transformer blocks sequentially apply gareko

        x = self.final_norm(x)
        # final normalization apply gareko

        logits = self.out_head(x)
        # output linear layer apply garera vocab logits banako

        return logits
        # final output (logits for each token position) return gareko

In [32]:
torch.manual_seed(123)
# random seed set gareko (reproducibility ko lagi)

model = GPTModel(GPT_CONFIG_124M)
# GPT model initialize gareko (124M config use garera)

out = model(batch)
# input batch lai model ma pathaera output logits nikaleko

print("Input batch:\n", batch)
# input batch print gareko

print("\nOutput shape:", out.shape)
# output tensor ko shape print gareko

print(out)
# final output logits print gareko

Input batch:
 tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])

Output shape: torch.Size([2, 4, 50257])
tensor([[[ 0.3613,  0.4222, -0.0711,  ...,  0.3483,  0.4661, -0.2838],
         [-0.1792, -0.5660, -0.9485,  ...,  0.0477,  0.5181, -0.3168],
         [ 0.7120,  0.0332,  0.1085,  ...,  0.1018, -0.4327, -0.2553],
         [-1.0076,  0.3418, -0.1190,  ...,  0.7195,  0.4023,  0.0532]],

        [[-0.2564,  0.0900,  0.0335,  ...,  0.2659,  0.4454, -0.6806],
         [ 0.1230,  0.3653, -0.2074,  ...,  0.7705,  0.2710,  0.2246],
         [ 1.0558,  1.0318, -0.2800,  ...,  0.6936,  0.3205, -0.3178],
         [-0.1565,  0.3926,  0.3288,  ...,  1.2630, -0.1858,  0.0388]]],
       grad_fn=<UnsafeViewBackward0>)


In [33]:
total_params = sum(p.numel() for p in model.parameters())
# model ko sabai parameters count gareko (numel = number of elements)

print(f"Total number of parameters: {total_params:,}")
# total parameters lai formatted string ma print gareko (comma separated)

Total number of parameters: 163,009,536


In [34]:
print("Token embedding layer shape:", model.tok_emb.weight.shape)
# token embedding layer ko weight matrix ko shape print gareko
# shape: [vocab_size, emb_dim]

print("Output layer shape:", model.out_head.weight.shape)
# output linear layer ko weight matrix ko shape print gareko
# shape: [emb_dim, vocab_size]

Token embedding layer shape: torch.Size([50257, 768])
Output layer shape: torch.Size([50257, 768])


In [35]:
total_params_gpt2 = total_params - sum(p.numel() for p in model.out_head.parameters())
# GPT-2 style weight tying consider gareko
# output layer ko parameters subtract gareko (embedding ra output layer share hunxa)

print(f"Number of trainable parameters considering weight tying: {total_params_gpt2:,}")
# weight tying use garda trainable parameters ko total count print gareko (comma separated)

Number of trainable parameters considering weight tying: 124,412,160


In [36]:
total_size_bytes = total_params * 4  # A
# total parameters * 4 bytes (float32 = 4 bytes per parameter)
# model ko total size bytes ma calculate gareko

total_size_mb = total_size_bytes / (1024 * 1024)  # B
# bytes lai megabytes ma convert gareko (divide by 1024*1024)

print(f"Total size of the model: {total_size_mb:.2f} MB")
# model ko total size MB ma formatted string ma print gareko (2 decimal places)

Total size of the model: 621.83 MB


## ya bata token haru lai output text ma convert garney process ho


In [37]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx = (batch, n_tokens) array of indices (current context)

    for _ in range(max_new_tokens):
        # loop garera naya tokens generate garna

        # context crop gareko if it exceeds context_size
        # e.g. LLM supports 5 tokens but context has 10 → last 5 tokens matra use hunxa
        idx_cond = idx[:, -context_size:]

        # model prediction nikaleko (no gradient calculation)
        with torch.no_grad():
            logits = model(idx_cond)

        # focus only on last time step
        # (batch, n_tokens, vocab_size) → (batch, vocab_size)
        logits = logits[:, -1, :]

        # softmax apply garera probability distribution banako
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # sabai bhanda highest probability ko token choose gareko
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)

        # naya token append gareko sequence ma
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx
    # final generated sequence return gareko

In [ ]:
start_context = "Hello, I am"  # change garara naya text generate garna sakinxa
# starting text string define gareko

encoded = tokenizer.encode(start_context)
# tokenizer use garera text lai token IDs ma convert gareko
print("encoded:", encoded)

encoded_tensor = torch.tensor(encoded).unsqueeze(0)  # A
# token IDs lai tensor ma convert gareko
# unsqueeze(0) → batch dimension add gareko (shape: [1, seq_len])

print("encoded_tensor.shape:", encoded_tensor.shape)
# encoded tensor ko shape print gareko

encoded: [15496, 11, 314, 716, 27018, 24086, 2750, 274, 16239, 42348, 7267]
encoded_tensor.shape: torch.Size([1, 11])


In [42]:
model.eval()  # A
# model lai evaluation mode ma set gareko (dropout, batchnorm disable hunxa)

out = generate_text_simple(
    model=model,
    idx=encoded_tensor,
    max_new_tokens=6,
    context_size=GPT_CONFIG_124M["context_length"],
)
# generate_text_simple function call gareko
# starting context bata 6 new tokens generate gareko
# context_size = model ko supported max length

print("Output:", out)
# generated output tensor print gareko

print("Output length:", len(out[0]))
# final sequence ko length print gareko (original + generated tokens)

Output: tensor([[15496,    11,   314,   716, 27018, 24086,  2750,   274, 16239, 42348,
          7267,  4003, 30379,  5259, 49402,   638, 20282]])
Output length: 17


In [ ]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
# output tensor lai squeeze(0) garera batch dimension hataeko
# tensor → list of token IDs convert gareko
# tokenizer.decode use garera token IDs lai back to text banako

print(decoded_text)
# final generated text print gareko

Hello, I am Featureiman Byeswickattribute argue notice Arlington tit refuted knstrom
